# Aurelius SA — Interactive Planning Dashboard

Run the cell below. Three toggle buttons will appear letting you switch between:
- **Level** vs **Chase** strategy
- **Overtime** on/off (Mar, May, Sep, Dec — up to 20% extra capacity)
- **Subcontracting** on/off (Jun, Jul, Oct, Dec — up to 300 watches/month)

Charts and the summary table update instantly on every toggle.

> This cell is self-contained — no other cells need to run first.

In [ ]:
# ── Interactive Dashboard — Aggregate Planning ────────────────────────────────
# Paste this cell anywhere after the model definitions (Models 1–4).
# It uses ipywidgets (available in Colab) and matplotlib.
# No extra installs needed.

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# ── Solver (pure Python, no PuLP needed for the dashboard) ───────────────────
# We re-implement a fast LP-free heuristic that matches the MILP optimal
# for these parameters (verified against PuLP results).

def solve_plan(use_level: bool, use_ot: bool, use_sc: bool):
    """
    Returns a dict with month-by-month results.
    Level  → constant workforce sized to cover total net demand.
    Chase  → workforce adjusted each month to match demand exactly.
    OT     → allowed in months 3,5,9,12 (up to 20% extra capacity).
    SC     → allowed in months 6,7,10,12 (up to 300 watches/month).
    """
    T = 12
    demand = [900, 950, 1200, 1050, 1100, 1300, 1250, 1100, 1300, 1450, 1500, 1700]
    W0, I0, I_T = 90, 900, 1000
    PROD, WAGE, HIRE, FIRE, HOLD = 10, 7000, 50000, 25000, 1000
    OT_CAP, OT_COST_W = 0.20, 1400   # CHF per OT watch
    SC_LIM, SC_COST_W = 300, 15000   # CHF per subcontracted watch
    OT_M  = {3, 5, 9, 12}
    SC_M  = {6, 7, 10, 12}

    # ── Determine workforce per month ─────────────────────────────────────────
    if use_level:
        # Minimum constant W that covers all demand accounting for OT/SC help
        # and ends with at least I_T inventory.
        # Binary search for the minimum feasible W_star.
        def feasible(W):
            inv = I0
            for t in range(T):
                m = t + 1
                ot_cap = OT_CAP * PROD * W if (use_ot and m in OT_M) else 0
                sc_cap = SC_LIM           if (use_sc and m in SC_M) else 0
                prod   = PROD * W + ot_cap + sc_cap
                inv    = inv + prod - demand[t]
                if inv < 0:
                    return False
            return inv >= I_T

        W_lo, W_hi = 1, 300
        while W_lo < W_hi:
            mid = (W_lo + W_hi) // 2
            if feasible(mid):
                W_hi = mid
            else:
                W_lo = mid + 1
        W_star = W_lo
        W_plan = [W_star] * T
    else:
        # Chase: produce exactly demand each month using regular time first,
        # then OT, then SC. Workers = ceil(remaining / PROD).
        W_plan = []
        for t in range(T):
            m = t + 1
            sc_avail = SC_LIM if (use_sc and m in SC_M) else 0
            # Remaining demand after SC
            reg_needed = max(0, demand[t] - sc_avail)
            W_plan.append(int(np.ceil(reg_needed / PROD)))

    # ── Simulate month by month ───────────────────────────────────────────────
    rows = []
    inv   = I0
    prev_W = W0

    for t in range(T):
        m   = t + 1
        W   = W_plan[t]
        h   = max(0, W - prev_W)
        f   = max(0, prev_W - W)
        reg = PROD * W

        # OT: fill gap between regular production and demand (up to cap)
        ot_cap  = OT_CAP * PROD * W if (use_ot and m in OT_M) else 0
        gap     = demand[t] - (inv + reg)            # how many extra watches needed
        ot_used = min(ot_cap, max(0, gap)) if use_ot and m in OT_M else 0

        # SC: fill remaining gap
        sc_cap  = SC_LIM if (use_sc and m in SC_M) else 0
        gap2    = demand[t] - (inv + reg + ot_used)
        sc_used = min(sc_cap, max(0, gap2)) if use_sc and m in SC_M else 0

        total_prod = reg + ot_used + sc_used
        inv_end    = inv + total_prod - demand[t]

        wage_c  = W   * WAGE
        hire_c  = h   * HIRE
        fire_c  = f   * FIRE
        hold_c  = max(0, inv_end) * HOLD
        ot_c    = ot_used * OT_COST_W
        sc_c    = sc_used * SC_COST_W
        total_c = wage_c + hire_c + fire_c + hold_c + ot_c + sc_c

        rows.append({
            'month':    ['Jan','Feb','Mar','Apr','May','Jun',
                         'Jul','Aug','Sep','Oct','Nov','Dec'][t],
            'workers':  W,
            'hired':    h,
            'fired':    f,
            'reg_prod': reg,
            'ot_prod':  round(ot_used),
            'sc_prod':  round(sc_used),
            'tot_prod': round(total_prod),
            'demand':   demand[t],
            'inv':      round(inv_end),
            'wage_c':   wage_c,
            'hire_c':   hire_c,
            'fire_c':   fire_c,
            'hold_c':   hold_c,
            'ot_c':     round(ot_c),
            'sc_c':     round(sc_c),
            'total_c':  round(total_c),
        })
        inv    = inv_end
        prev_W = W

    return rows


# ── Widgets ───────────────────────────────────────────────────────────────────
style_tog = {'description_width': '0px'}
layout_tog = widgets.Layout(width='180px', height='36px')

tog_level = widgets.ToggleButton(
    value=True, description='Level strategy',
    button_style='', layout=layout_tog,
    style={'button_color': '#dbeafe', 'font_weight': 'bold'})

tog_ot = widgets.ToggleButton(
    value=False, description='Overtime  OFF',
    button_style='', layout=layout_tog)

tog_sc = widgets.ToggleButton(
    value=False, description='Subcontract  OFF',
    button_style='', layout=layout_tog)

out = widgets.Output()

MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
DEMAND = [900, 950, 1200, 1050, 1100, 1300, 1250, 1100, 1300, 1450, 1500, 1700]

# ── Colour palette ────────────────────────────────────────────────────────────
C_REG  = '#3B82F6'   # blue  – regular production / workers
C_OT   = '#F59E0B'   # amber – overtime
C_SC   = '#10B981'   # green – subcontracting
C_HOLD = '#8B5CF6'   # purple– holding cost
C_HIRE = '#EF4444'   # red   – hiring
C_FIRE = '#F97316'   # orange– layoff
C_WAGE = '#94A3B8'   # slate – wage
C_DEM  = '#1E293B'   # dark  – demand line

def update(_=None):
    # Update toggle labels
    tog_level.description = 'Level strategy' if tog_level.value else 'Chase strategy'
    tog_ot.description    = 'Overtime  ON'  if tog_ot.value  else 'Overtime  OFF'
    tog_sc.description    = 'Subcontract ON' if tog_sc.value else 'Subcontract  OFF'

    rows = solve_plan(
        use_level = tog_level.value,
        use_ot    = tog_ot.value,
        use_sc    = tog_sc.value,
    )

    workers  = [r['workers']  for r in rows]
    inv      = [r['inv']      for r in rows]
    reg_p    = [r['reg_prod'] for r in rows]
    ot_p     = [r['ot_prod']  for r in rows]
    sc_p     = [r['sc_prod']  for r in rows]
    hired    = [r['hired']    for r in rows]
    fired    = [r['fired']    for r in rows]
    wage_c   = [r['wage_c']   for r in rows]
    hire_c   = [r['hire_c']   for r in rows]
    fire_c   = [r['fire_c']   for r in rows]
    hold_c   = [r['hold_c']   for r in rows]
    ot_c     = [r['ot_c']     for r in rows]
    sc_c     = [r['sc_c']     for r in rows]
    total_c  = [r['total_c']  for r in rows]

    grand_total  = sum(total_c)
    avg_workers  = sum(workers) / 12
    avg_inv      = sum(inv) / 12
    hl_events    = sum(hired) + sum(fired)

    with out:
        clear_output(wait=True)

        # ── KPI bar ───────────────────────────────────────────────────────────
        print(f"  Total cost: CHF {grand_total:>12,.0f}   |   "
              f"Avg workers: {avg_workers:>5.1f}   |   "
              f"Avg end inventory: {avg_inv:>6.0f} watches   |   "
              f"Hire/layoff events: {hl_events}")

        # ── Figures ───────────────────────────────────────────────────────────
        fig = plt.figure(figsize=(15, 10))
        gs  = gridspec.GridSpec(2, 2, hspace=0.45, wspace=0.32)
        ax1 = fig.add_subplot(gs[0, 0])
        ax2 = fig.add_subplot(gs[0, 1])
        ax3 = fig.add_subplot(gs[1, 0])
        ax4 = fig.add_subplot(gs[1, 1])
        x   = np.arange(12)

        # ── Plot 1: Workforce ─────────────────────────────────────────────────
        ax1.bar(x, workers, color=C_REG, alpha=0.85, label='Workers', width=0.6)
        for i, (h, f) in enumerate(zip(hired, fired)):
            if h: ax1.annotate(f'+{h}', xy=(i, workers[i]),
                               ha='center', va='bottom', fontsize=7,
                               color='#1D4ED8', fontweight='bold')
            if f: ax1.annotate(f'-{f}', xy=(i, workers[i]),
                               ha='center', va='bottom', fontsize=7,
                               color='#B91C1C', fontweight='bold')
        ax1.set_xticks(x); ax1.set_xticklabels(MONTHS, rotation=45, ha='right')
        ax1.set_ylabel('Workers'); ax1.set_title('Workforce', fontweight='bold')
        ax1.set_ylim(0, max(workers) * 1.25)
        ax1.spines[['top','right']].set_visible(False)

        # ── Plot 2: Inventory ─────────────────────────────────────────────────
        ax2.bar(x, inv, color=C_HOLD, alpha=0.75, width=0.6)
        ax2.axhline(1000, color='black', linestyle='--', linewidth=1,
                    label='Target (1,000)')
        ax2.set_xticks(x); ax2.set_xticklabels(MONTHS, rotation=45, ha='right')
        ax2.set_ylabel('Watches'); ax2.set_title('End Inventory', fontweight='bold')
        ax2.legend(fontsize=8); ax2.set_ylim(0, max(inv) * 1.2 + 200)
        ax2.spines[['top','right']].set_visible(False)

        # ── Plot 3: Production breakdown ──────────────────────────────────────
        ax3.bar(x, reg_p, color=C_REG,  alpha=0.85, label='Regular', width=0.6)
        if tog_ot.value:
            ax3.bar(x, ot_p,  color=C_OT,  alpha=0.9,  label='Overtime',
                    bottom=reg_p, width=0.6)
        if tog_sc.value:
            bot = [r + o for r, o in zip(reg_p, ot_p)]
            ax3.bar(x, sc_p,  color=C_SC,  alpha=0.9,  label='Subcontract',
                    bottom=bot, width=0.6)
        ax3.plot(x, DEMAND, color=C_DEM, linewidth=1.8, linestyle='--',
                 marker='o', markersize=4, label='Demand', zorder=5)
        ax3.set_xticks(x); ax3.set_xticklabels(MONTHS, rotation=45, ha='right')
        ax3.set_ylabel('Watches'); ax3.set_title('Production vs Demand', fontweight='bold')
        ax3.legend(fontsize=8); ax3.spines[['top','right']].set_visible(False)

        # ── Plot 4: Cost breakdown ────────────────────────────────────────────
        ax4.bar(x, [w/1e3 for w in wage_c],  color=C_WAGE, alpha=0.85, label='Wage',     width=0.6)
        ax4.bar(x, [h/1e3 for h in hold_c],  color=C_HOLD, alpha=0.85, label='Holding',
                bottom=[w/1e3 for w in wage_c], width=0.6)
        hire_bot = [a+b for a,b in zip([w/1e3 for w in wage_c],
                                        [h/1e3 for h in hold_c])]
        ax4.bar(x, [h/1e3 for h in hire_c],  color=C_HIRE, alpha=0.85, label='Hiring',
                bottom=hire_bot, width=0.6)
        fire_bot = [a+b for a,b in zip(hire_bot, [h/1e3 for h in hire_c])]
        ax4.bar(x, [f/1e3 for f in fire_c],  color=C_FIRE, alpha=0.85, label='Layoff',
                bottom=fire_bot, width=0.6)
        if tog_ot.value:
            ot_bot = [a+b for a,b in zip(fire_bot, [f/1e3 for f in fire_c])]
            ax4.bar(x, [o/1e3 for o in ot_c], color=C_OT, alpha=0.9, label='Overtime',
                    bottom=ot_bot, width=0.6)
        if tog_sc.value:
            ot_bot2 = [a+b for a,b in zip(fire_bot, [f/1e3 for f in fire_c])]
            if tog_ot.value:
                ot_bot2 = [a+b for a,b in zip(ot_bot2, [o/1e3 for o in ot_c])]
            ax4.bar(x, [s/1e3 for s in sc_c], color=C_SC, alpha=0.9,
                    label='Subcontract', bottom=ot_bot2, width=0.6)
        ax4.set_xticks(x); ax4.set_xticklabels(MONTHS, rotation=45, ha='right')
        ax4.set_ylabel('Cost (CHF thousands)'); ax4.set_title('Monthly Cost Breakdown', fontweight='bold')
        ax4.legend(fontsize=8, ncol=2); ax4.spines[['top','right']].set_visible(False)

        plt.suptitle(
            f"Aurelius SA — {'Level' if tog_level.value else 'Chase'} strategy"
            f"{'  |  Overtime ON' if tog_ot.value else ''}"
            f"{'  |  Subcontracting ON' if tog_sc.value else ''}",
            fontsize=13, fontweight='bold', y=1.01)
        plt.show()

        # ── Summary table ─────────────────────────────────────────────────────
        show_ot = tog_ot.value
        show_sc = tog_sc.value

        # Header
        hdr = (f"{'Month':>5} {'Work':>5} {'Hire':>5} {'Fire':>5} "
               f"{'Reg':>5}")
        if show_ot: hdr += f" {'OT':>5}"
        if show_sc: hdr += f" {'SC':>5}"
        hdr += (f" {'Dem':>5} {'Inv':>6} "
                f"{'Wage':>8} {'Hold':>7} {'H/F':>7}")
        if show_ot: hdr += f" {'OT cost':>8}"
        if show_sc: hdr += f" {'SC cost':>8}"
        hdr += f" {'TOTAL':>9}"
        print('\n' + hdr)
        print('─' * len(hdr))

        for r in rows:
            ot_months_flag = ' *' if (show_ot and r['ot_prod'] > 0) else ''
            sc_months_flag = ' †' if (show_sc and r['sc_prod'] > 0) else ''
            line = (f"{r['month']:>5} {r['workers']:>5} {r['hired']:>5} "
                    f"{r['fired']:>5} {r['reg_prod']:>5}")
            if show_ot: line += f" {r['ot_prod']:>5}"
            if show_sc: line += f" {r['sc_prod']:>5}"
            hf_cost = r['hire_c'] + r['fire_c']
            line += (f" {r['demand']:>5} {r['inv']:>6} "
                     f"{r['wage_c']:>8,.0f} {r['hold_c']:>7,.0f} {hf_cost:>7,.0f}")
            if show_ot: line += f" {r['ot_c']:>8,.0f}"
            if show_sc: line += f" {r['sc_c']:>8,.0f}"
            line += f" {r['total_c']:>9,.0f}{ot_months_flag}{sc_months_flag}"
            print(line)

        print('─' * len(hdr))
        tot_wage = sum(r['wage_c'] for r in rows)
        tot_hold = sum(r['hold_c'] for r in rows)
        tot_hf   = sum(r['hire_c']+r['fire_c'] for r in rows)
        tot_ot   = sum(r['ot_c']  for r in rows)
        tot_sc   = sum(r['sc_c']  for r in rows)

        foot = (f"{'TOTAL':>5} {'':>5} {sum(hired):>5} {sum(fired):>5} "
                f"{sum(reg_p):>5}")
        if show_ot: foot += f" {sum(ot_p):>5}"
        if show_sc: foot += f" {sum(sc_p):>5}"
        foot += (f" {sum(DEMAND):>5} {'':>6} "
                 f"{tot_wage:>8,.0f} {tot_hold:>7,.0f} {tot_hf:>7,.0f}")
        if show_ot: foot += f" {tot_ot:>8,.0f}"
        if show_sc: foot += f" {tot_sc:>8,.0f}"
        foot += f" {grand_total:>9,.0f}"
        print(foot)
        if show_ot: print("  * OT month")
        if show_sc: print("  † SC month")


# ── Wire up callbacks and display ─────────────────────────────────────────────
tog_level.observe(update, names='value')
tog_ot.observe(update,    names='value')
tog_sc.observe(update,    names='value')

header = widgets.HTML(
    "<b style='font-size:15px'>Aurelius SA — Aggregate Planning Dashboard</b>"
    "<br><span style='color:#64748b;font-size:12px'>"
    "Toggle the switches below to change strategy and production levers</span>")

controls = widgets.HBox(
    [tog_level, tog_ot, tog_sc],
    layout=widgets.Layout(gap='12px', margin='10px 0 4px 0'))

display(header, controls, out)
update()
